In [ ]:
import os, random, math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

SEED = 7
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

## 1) Daten laden

In [ ]:
CSV_PATH = "free_throw_dataset.csv"
if not os.path.exists(CSV_PATH):
    alt = os.path.join("/mnt/data", CSV_PATH)
    if os.path.exists(alt):
        CSV_PATH = alt

df = pd.read_csv(CSV_PATH)
print("CSV_PATH:", CSV_PATH)
print(df.shape)
print(df.columns.tolist())
df.head()

In [ ]:
per_throw = df.groupby("wurf_id").agg({
    "label":"first",
    "theta":"first",
    "v0":"first",
    "T":"first"
}).reset_index()

print("Anzahl Würfe:", per_throw.shape[0])
print(per_throw["label"].value_counts())
per_throw.head()

## 2) Train/Val/Test Split (nach `wurf_id`, stratifiziert nach Label)

In [ ]:
train_ids, test_ids = train_test_split(
    per_throw["wurf_id"].values,
    test_size=0.15,
    random_state=SEED,
    stratify=per_throw["label"].values,
)
train_ids, val_ids = train_test_split(
    train_ids,
    test_size=0.15,
    random_state=SEED,
    stratify=per_throw.set_index("wurf_id").loc[train_ids, "label"].values,
)

print("Train/Val/Test:", len(train_ids), len(val_ids), len(test_ids))
print("Alle Test ids:", test_ids)

## 3) Dataset: gekürzte Beobachtungen + Ground Truth

In [ ]:
class FreeThrowDataset(Dataset):
    """
    Liefert pro Wurf die ersten n_obs Beobachtungen (t_obs, xy_obs) und die komplette Trajektorie (t_full, xy_full).

    Wichtig: kein Zukunftswissen via T/t_max – stattdessen liefern wir t_min, t_obs_max und denom = t_obs_max - t_min
    fuer eine zeitliche Normalisierung relativ zum Beobachtungsfenster.
    """
    def __init__(self, df: pd.DataFrame, throw_ids, n_obs: int = 30, use_noisy: bool = False):
        self.n_obs = n_obs
        self.use_noisy = use_noisy

        sub = df[df["wurf_id"].isin(throw_ids)].copy()
        sub.sort_values(["wurf_id", "t"], inplace=True)

        self.throw_ids = np.array(sorted(sub["wurf_id"].unique()))
        self.groups = {tid: g for tid, g in sub.groupby("wurf_id")}

        if use_noisy:
            self.x_col, self.y_col = "x_noisy", "y_noisy"
        else:
            self.x_col, self.y_col = "x_clean", "y_clean"

    def __len__(self):
        return len(self.throw_ids)

    def __getitem__(self, idx):
        tid = int(self.throw_ids[idx])
        g = self.groups[tid]

        t = torch.tensor(g["t"].values, dtype=torch.float32)  # (M,)
        xy = torch.tensor(
            np.stack([g[self.x_col].values, g[self.y_col].values], axis=1),
            dtype=torch.float32
        )  # (M,2)

        t_obs = t[:self.n_obs]
        xy_obs = xy[:self.n_obs]

        theta = torch.tensor(float(g["theta"].iloc[0]), dtype=torch.float32)
        v0 = torch.tensor(float(g["v0"].iloc[0]), dtype=torch.float32)
        label = torch.tensor(int(g["label"].iloc[0]), dtype=torch.long)

        # Zeit-Normalisierung nur ueber beobachtetes Fenster:
        t_min = t_obs[0]
        t_obs_max = t_obs[-1]
        denom = (t_obs_max - t_min).clamp_min(1e-4)

        return {
            "wurf_id": tid,
            "t_obs": t_obs,          # (n_obs,)
            "xy_obs": xy_obs,        # (n_obs,2)
            "t_full": t,             # (M,)
            "xy_full": xy,           # (M,2)
            "t_min": t_min,          # ()
            "t_obs_max": t_obs_max,  # ()
            "denom": denom,          # ()
            "theta": theta,
            "v0": v0,
            "label": label,
        }

def collate_fn(batch):
    out = {}
    for k in batch[0].keys():
        if k == "wurf_id":
            out[k] = [b[k] for b in batch]
        else:
            out[k] = torch.stack([b[k] for b in batch], dim=0)
    return out

n_obs = 30
train_ds = FreeThrowDataset(df, train_ids, n_obs=n_obs, use_noisy=False)
val_ds   = FreeThrowDataset(df, val_ids,   n_obs=n_obs, use_noisy=False)
test_ds  = FreeThrowDataset(df, test_ids,  n_obs=n_obs, use_noisy=False)

train_loader = DataLoader(train_ds, batch_size=64, shuffle=True, collate_fn=collate_fn)
val_loader   = DataLoader(val_ds,   batch_size=64, shuffle=False, collate_fn=collate_fn)
test_loader  = DataLoader(test_ds,  batch_size=64, shuffle=False, collate_fn=collate_fn)

batch = next(iter(train_loader))
{k: (v.shape if torch.is_tensor(v) else len(v)) for k,v in batch.items()}

In [ ]:
# === Plot: Einzelbeispiel (Clean vs. Noisy) + Beobachtungsfenster ===
# Wähle ein Beispiel (zufällig, aber reproduzierbar über SEED)
example = per_throw.sample(1, random_state=SEED).iloc[0]
wurf_id = int(example["wurf_id"])
label = int(example["label"])

g = df[df["wurf_id"] == wurf_id].sort_values("t").reset_index(drop=True)

n_obs = 30
n_obs = min(n_obs, len(g))

plt.figure(figsize=(7, 5))

# (i) rauschfreie Trajektorie als Kurve
plt.plot(g["x_clean"].values, g["y_clean"].values, label="Rauschfrei (Referenz)")

# (ii) verrauschte Stützstellen
plt.scatter(
    g["x_noisy"].values,
    g["y_noisy"].values,
    s=18,
    alpha=0.7,
    label="verrauscht (Beobachtung)",
)

# Markierung des Beobachtungspräfix (erste n_obs Punkte)
plt.scatter(
    g.loc[: n_obs - 1, "x_noisy"].values,
    g.loc[: n_obs - 1, "y_noisy"].values,
    s=40,
    marker="s",
    label=f"Beobachtungen (erste {n_obs})",
)

plt.xlabel("x (m)")
plt.ylabel("y (m)")
plt.title(f"Rauschfreie Referenz vs. verrauschte Beobachtung mit Beobachtungsfenster")
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

## 4) Modell: Encoder → (θ,v0) + Decoder (x,y) + Klassifikations-Head

In [ ]:
class MLP(nn.Module):
    def __init__(self, in_dim: int, out_dim: int, hidden: int = 128, depth: int = 3, dropout: float = 0.0):
        super().__init__()
        layers = []
        d = in_dim
        for _ in range(depth):
            layers += [nn.Linear(d, hidden), nn.Tanh()]
            if dropout > 0:
                layers += [nn.Dropout(dropout)]
            d = hidden
        layers += [nn.Linear(d, out_dim)]
        self.net = nn.Sequential(*layers)
    def forward(self, x):
        return self.net(x)

class AmortizedPINNClassifier(nn.Module):
    """
    Encoder (GRU) ueber beobachtete Sequenz -> Latent z.
    Heads: theta, v0, Klassifikation.
    Decoder: amortisiertes PINN, bekommt KEIN T/t_max, sondern
    - normierte Zeit t_n
    - log_denom = log(t_obs_max - t_min) als Skalierungskontext
    """
    def __init__(
        self,
        enc_hidden: int = 128,
        z_dim: int = 128,
        dec_hidden: int = 128,
        num_classes: int = 3,
        theta_min: float = 0.5,
        theta_max: float = 1.2,
        v0_min: float = 0.0
    ):
        super().__init__()
        self.theta_min = theta_min
        self.theta_max = theta_max
        self.v0_min = v0_min

        # Encoder input: (t_n, x, y)
        self.in_proj = nn.Linear(3, enc_hidden)
        self.gru = nn.GRU(input_size=enc_hidden, hidden_size=enc_hidden, batch_first=True)
        self.to_z = nn.Linear(enc_hidden, z_dim)

        self.theta_head = MLP(z_dim, 1, hidden=128, depth=2)
        self.v0_head    = MLP(z_dim, 1, hidden=128, depth=2)

        # Decoder input: z + (t_n, log_denom)
        self.decoder = MLP(z_dim + 2, 2, hidden=dec_hidden, depth=3)

        self.classifier = MLP(z_dim + 2, num_classes, hidden=128, depth=2, dropout=0.1)

    def encode(self, t_n_obs, xy_obs):
        inp = torch.cat([t_n_obs.unsqueeze(-1), xy_obs], dim=-1)  # (B,n,3)
        h = torch.tanh(self.in_proj(inp))
        _, hT = self.gru(h)  # (1,B,H)
        z = self.to_z(hT.squeeze(0))
        return z

    def predict_params(self, z):
        theta_raw = self.theta_head(z).squeeze(-1)
        v0_raw = self.v0_head(z).squeeze(-1)
        theta = self.theta_min + (self.theta_max - self.theta_min) * torch.sigmoid(theta_raw)
        v0 = self.v0_min + F.softplus(v0_raw)
        return theta, v0

    def decode_traj(self, z, t_n, log_denom):
        B, N = t_n.shape
        logd = log_denom.unsqueeze(1).expand(-1, N)  # [B,N]
        inp = torch.cat([
            z.unsqueeze(1).expand(-1, N, -1),
            t_n.unsqueeze(-1),
            logd.unsqueeze(-1),
        ], dim=-1)
        return self.decoder(inp)

    def forward(self, t_n_obs, xy_obs, t_n_query, log_denom):
        z = self.encode(t_n_obs, xy_obs)
        theta, v0 = self.predict_params(z)
        xy_hat = self.decode_traj(z, t_n_query, log_denom)
        logits = self.classifier(torch.cat([z, theta.unsqueeze(-1), v0.unsqueeze(-1)], dim=-1))
        return {"z": z, "theta": theta, "v0": v0, "xy_hat": xy_hat, "logits": logits}

model = AmortizedPINNClassifier(num_classes=int(df["label"].nunique())).to(device)
sum(p.numel() for p in model.parameters())

## 5) Loss: Daten + Physik + IC + Klassifikation (+ Param-Supervision)

In [ ]:
G = 9.81
H_RIM = 3.048  # 10 ft in meters

def normalize_time(t, t_min, denom):
    # t: [B,N] oder [B]
    if t.dim() == 1:
        return (t - t_min) / denom
    return (t - t_min.unsqueeze(1)) / denom.unsqueeze(1)

def denormalize_time(t_n, t_min, denom):
    # t_n: [B,N]
    return t_min.unsqueeze(1) + t_n * denom.unsqueeze(1)

def time_to_height(theta, v0, y0, h=H_RIM, g=G, eps=1e-9):
    a = 0.5 * g
    b = -v0 * torch.sin(theta)
    c = h - y0

    disc = b*b - 4*a*c
    disc_clamped = torch.clamp(disc, min=0.0)
    sqrt_disc = torch.sqrt(disc_clamped + eps)

    t1 = (-b - sqrt_disc) / (2*a)
    t2 = (-b + sqrt_disc) / (2*a)

    # spaetere Loesung (typisch Abstieg)
    t_hit = torch.maximum(t1, t2)
    valid = (disc > 0) & (t_hit > 0)
    return t_hit, valid

def make_query_times_until_rim_height(
    t_min, t_obs_max, denom, theta, v0, y0,
    dt, h_rim=H_RIM,
    t_end_min_offset=0.0,
    t_end_max_offset=3.0
):
    # robustes dt
    dt = float(dt) if dt is not None else 0.02
    dt = max(dt, 1e-4)

    t_hit_rel, valid = time_to_height(theta, v0, y0, h=h_rim)
    t_hit_abs = t_min + t_hit_rel

    t_end_raw = torch.where(valid, t_hit_abs, t_obs_max + t_end_max_offset)
    t_end = torch.clamp(
        t_end_raw,
        min=t_obs_max + t_end_min_offset,
        max=t_obs_max + t_end_max_offset
    )

    # grid <= t_end, plus ggf. exakter Endpunkt
    t_min_val = float(t_min.item())
    t_end_val = float(t_end.item())

    N = int(math.floor((t_end_val - t_min_val) / dt)) + 1
    N = max(N, 2)
    t = t_min_val + torch.arange(N, device=t_min.device, dtype=t_min.dtype) * dt

    # keine Überschreitung: ggf. letzten Punkt auf t_end setzen oder t_end anhängen
    if float(t[-1].item()) > t_end_val + 1e-9:
        t[-1] = t_end.squeeze()
    elif float(t[-1].item()) < t_end_val - 1e-9:
        t = torch.cat([t, t_end.squeeze().unsqueeze(0)], dim=0)

    return t.unsqueeze(0)  # [1,N]


def trim_prediction_at_rim_height(t_query_1d, xy_hat_2d, h_rim=H_RIM, eps=1e-9):
    y = xy_hat_2d[:, 1]
    above = y >= (h_rim - eps)

    if not torch.any(above):
        return t_query_1d, xy_hat_2d

    last_above = int(torch.where(above)[0][-1].item())
    if last_above >= (len(y) - 1):
        return t_query_1d[: last_above + 1], xy_hat_2d[: last_above + 1]

    # Interpolation zwischen last_above und next
    y1, y2 = y[last_above], y[last_above + 1]
    dy = (y2 - y1)
    if torch.abs(dy) < 1e-12:
        return t_query_1d[: last_above + 1], xy_hat_2d[: last_above + 1]

    alpha = (h_rim - y1) / dy
    alpha = torch.clamp(alpha, 0.0, 1.0)

    x1, x2 = xy_hat_2d[last_above, 0], xy_hat_2d[last_above + 1, 0]
    t1, t2 = t_query_1d[last_above], t_query_1d[last_above + 1]

    x_cross = x1 + alpha * (x2 - x1)
    t_cross = t1 + alpha * (t2 - t1)

    xy_cross = torch.stack([
        x_cross,
        torch.tensor(h_rim, device=xy_hat_2d.device, dtype=xy_hat_2d.dtype)
    ], dim=0)

    t_new = torch.cat([t_query_1d[: last_above + 1], t_cross.unsqueeze(0)], dim=0)
    xy_new = torch.cat([xy_hat_2d[: last_above + 1], xy_cross.unsqueeze(0)], dim=0)

    return t_new, xy_new



def physics_residuals_time_normed(out, t_n, t_min, denom):
    xy = out["xy_hat"]
    x = xy[..., 0]
    y = xy[..., 1]
    theta = out["theta"]
    v0 = out["v0"]

    ones = torch.ones_like(x)
    dx_dt_n = torch.autograd.grad(x, t_n, grad_outputs=ones, create_graph=True, retain_graph=True)[0]
    dy_dt_n = torch.autograd.grad(y, t_n, grad_outputs=ones, create_graph=True, retain_graph=True)[0]

    # Kettenregel: t_n = (t - t_min)/denom  => dt_n/dt = 1/denom
    dx_dt = dx_dt_n / denom.unsqueeze(1)
    dy_dt = dy_dt_n / denom.unsqueeze(1)

    # Physik nutzt t_rel = t - t_min (Zeit seit Beginn der Observations)
    t_phys = denormalize_time(t_n, t_min, denom)
    t_rel = t_phys - t_min.unsqueeze(1)

    target_dx = v0.unsqueeze(1) * torch.cos(theta).unsqueeze(1)
    target_dy = v0.unsqueeze(1) * torch.sin(theta).unsqueeze(1) - G * t_rel

    rx = dx_dt - target_dx
    ry = dy_dt - target_dy
    return rx, ry

def compute_losses(
    batch, model,
    n_col=64,
    h_rim=H_RIM,
    t_end_min_offset=0.25,
    t_end_max_offset=3.0,
    w_data=15.0, w_phys=0.5, w_ic=10.0, w_cls=0.01
):
    t_obs   = batch["t_obs"].to(device)      # [B,n_obs]
    xy_obs  = batch["xy_obs"].to(device)     # [B,n_obs,2]
    label   = batch["label"].to(device)

    t_min   = batch["t_min"].to(device)          # [B]
    t_obs_max = batch["t_obs_max"].to(device)    # [B]
    denom   = batch["denom"].to(device).clamp_min(1e-4)  # [B]
    log_denom = torch.log(denom)

    # normalisierte Zeit: Beobachtungen in [0,1], Zukunft bei >1
    t_n_obs = normalize_time(t_obs, t_min, denom)

    # Data + Classification auf Beobachtungen
    out_obs = model(t_n_obs, xy_obs, t_n_obs, log_denom)
    data_loss = F.mse_loss(out_obs["xy_hat"], xy_obs)

    # IC bei t_n=0 (t=t_min): passt den ersten Punkt an (robuster als x(0)=0 Annahme)
    t0_n = torch.zeros((t_obs.shape[0], 1), device=device)
    out0 = model(t_n_obs, xy_obs, t0_n, log_denom)
    ic_loss = F.mse_loss(out0["xy_hat"].squeeze(1), xy_obs[:, 0, :])

    # Dynamisches Collocation-Ende bis Korbhöhe (ballistisch) + Sicherheitsfenster
    y0 = xy_obs[:, 0, 1]  # Starthöhe aus Beobachtung (kein Oracle)
    theta = out_obs["theta"]
    v0 = out_obs["v0"]

    t_hit_rel, valid = time_to_height(theta, v0, y0, h=h_rim)
    t_hit_abs = t_min + t_hit_rel

    t_end_raw = torch.where(valid, t_hit_abs, t_obs_max + t_end_max_offset)
    t_end = torch.clamp(
        t_end_raw,
        min=t_obs_max + t_end_min_offset,
        max=t_obs_max + t_end_max_offset
    ).detach()  # detach gegen "Cheating" ueber t_end

    # Collocation-Zeiten pro Sample: t ~ U(t_min, t_end)
    B = t_obs.shape[0]
    u = torch.rand((B, n_col), device=device)
    t_col = t_min.unsqueeze(1) + u * (t_end.unsqueeze(1) - t_min.unsqueeze(1))  # [B,n_col]
    t_n_col = normalize_time(t_col, t_min, denom)                                # [B,n_col]
    t_n_col.requires_grad_(True)

    out_col = model(t_n_obs, xy_obs, t_n_col, log_denom)
    rx, ry = physics_residuals_time_normed(out_col, t_n_col, t_min, denom)
    phys_loss = rx.pow(2).mean() + ry.pow(2).mean()

    # Classification
    cls_loss = F.cross_entropy(out_obs["logits"], label)

    total = w_data*data_loss + w_ic*ic_loss + w_phys*phys_loss + w_cls*cls_loss

    return total, {
        "total": total.detach().item(),
        "data": data_loss.detach().item(),
        "ic": ic_loss.detach().item(),
        "phys": phys_loss.detach().item(),
        "cls": cls_loss.detach().item(),
    }

def compute_mse_obs_future(pred_xy, gt_xy, n_obs: int):
    mse_obs = F.mse_loss(pred_xy[:, :n_obs, :], gt_xy[:, :n_obs, :], reduction="mean")
    mse_future = F.mse_loss(pred_xy[:, n_obs:, :], gt_xy[:, n_obs:, :], reduction="mean")
    return mse_obs.item(), mse_future.item()

## 6) Training (mit Early Stopping)

In [ ]:
def run_epoch(loader, model, optimizer=None, **loss_kwargs):
    is_train = optimizer is not None
    model.train(is_train)
    meters = {"total":0, "data":0, "ic":0, "phys":0, "cls":0}
    n_batches = 0

    for batch in loader:
        if is_train:
            optimizer.zero_grad(set_to_none=True)

        total, parts = compute_losses(batch, model, **loss_kwargs)

        if is_train:
            total.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

        for k in meters:
            meters[k] += parts[k]
        n_batches += 1

    for k in meters:
        meters[k] /= max(n_batches,1)
    return meters

def train_model(model, train_loader, val_loader, epochs=800, lr=2e-3, patience=50, **loss_kwargs):
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    best = float("inf")
    best_state = None
    bad = 0
    history = {"train":[], "val":[]}

    for epoch in range(1, epochs+1):
        tr = run_epoch(train_loader, model, optimizer=optimizer, **loss_kwargs)
        va = run_epoch(val_loader, model, optimizer=None, **loss_kwargs)
        history["train"].append(tr)
        history["val"].append(va)

        if va["total"] < best - 1e-5:
            best = va["total"]
            best_state = {k: v.detach().cpu().clone() for k,v in model.state_dict().items()}
            bad = 0
        else:
            bad += 1

        if epoch % 10 == 0 or epoch == 1:
            print(f"Epoch {epoch:4d} | train {tr['total']:.4f} | val {va['total']:.4f} "
                  f"(data {va['data']:.4f}, phys {va['phys']:.4f}, cls {va['cls']:.4f})")

        if bad >= patience:
            print("Early stopping.")
            break

    if best_state is not None:
        model.load_state_dict(best_state)
    return history

history = train_model(
    model,
    train_loader,
    val_loader,
    epochs=800,
    lr=2e-3,
    patience=50,
    n_col=64,
    w_data=5.0,
    w_phys=1.0,
    w_ic=10.0,
    w_cls=0.01,
)

In [ ]:
def plot_history(history, key="total"):
    tr = [h[key] for h in history["train"]]
    va = [h[key] for h in history["val"]]
    plt.figure()
    plt.plot(tr, label=f"train {key}")
    plt.plot(va, label=f"val {key}")
    plt.xlabel("epoch"); plt.ylabel(key); plt.legend(); plt.grid(True, alpha=0.3)
    plt.show()

for k in ["total","data","phys","ic","cls"]:
    plot_history(history, k)

## 7) Evaluation auf Testset

In [ ]:
def plot_confusion_matrix(cm, title="Confusion Matrix", ax=None):
    """
    Plottet eine Confusion Matrix wie im Screenshot (imshow + Zahlen),
    mit frei übergebbarem Titel.
    """
    if ax is None:
        fig, ax = plt.subplots(figsize=(5, 4))
    else:
        fig = ax.figure

    im = ax.imshow(cm, interpolation="nearest")  # Standard-Colormap wie im Screenshot

    # Achsenbeschriftungen
    ax.set_title(title)
    ax.set_xlabel("Vorhergesagte Klasse")
    ax.set_ylabel("Wahre Klasse")

    # Ticks 0..C-1
    C = cm.shape[0]
    ax.set_xticks(np.arange(C))
    ax.set_yticks(np.arange(C))
    ax.set_xticklabels([str(i) for i in range(C)])
    ax.set_yticklabels([str(i) for i in range(C)])

    # Zahlen in die Zellen
    for i in range(C):
        for j in range(C):
            ax.text(j, i, str(int(cm[i, j])), ha="center", va="center", color="black")

    # verhindert abgeschnittene Achsen bei manchen Backends
    ax.set_ylim(C - 0.5, -0.5)

    fig.tight_layout()
    return fig, ax


@torch.no_grad()
def evaluate(loader, model, titel):
    model.eval()
    y_true, y_pred = [], []
    theta_true, theta_pred = [], []
    v0_true, v0_pred = [], []

    mse_obs_list = []
    mse_future_list = []
    mse_full_list = []

    for batch in loader:
        t_obs  = batch["t_obs"].to(device)      # [B,n_obs]
        xy_obs = batch["xy_obs"].to(device)     # [B,n_obs,2]
        t_full = batch["t_full"].to(device)     # [B,M]
        xy_full = batch["xy_full"].to(device)   # [B,M,2]

        t_min = batch["t_min"].to(device)       # [B]
        denom = batch["denom"].to(device).clamp_min(1e-4)  # [B]
        log_denom = torch.log(denom)

        t_n_obs  = normalize_time(t_obs, t_min, denom)     # [B,n_obs]
        t_n_full = normalize_time(t_full, t_min, denom)    # [B,M]

        out = model(t_n_obs, xy_obs, t_n_full, log_denom)

        pred = out["logits"].argmax(dim=1).cpu().numpy()
        y_pred.extend(pred.tolist())
        y_true.extend(batch["label"].numpy().tolist())

        theta_pred.extend(out["theta"].cpu().numpy().tolist())
        v0_pred.extend(out["v0"].cpu().numpy().tolist())
        theta_true.extend(batch["theta"].numpy().tolist())
        v0_true.extend(batch["v0"].numpy().tolist())

        mse_full = F.mse_loss(out["xy_hat"], xy_full, reduction="mean").item()
        mse_full_list.append(mse_full)

        mse_obs, mse_future = compute_mse_obs_future(out["xy_hat"], xy_full, n_obs=n_obs)
        mse_obs_list.append(mse_obs)
        mse_future_list.append(mse_future)

    acc = accuracy_score(y_true, y_pred)
    print("Accuracy:", acc)
    print("\nClassification report:\n", classification_report(y_true, y_pred))
    plot_confusion_matrix(
        confusion_matrix(y_true, y_pred),
        title=titel
    )

    theta_mae = np.mean(np.abs(np.array(theta_pred) - np.array(theta_true)))
    v0_mae = np.mean(np.abs(np.array(v0_pred) - np.array(v0_true)))
    print(f"theta MAE:    {theta_mae:.5f} rad")
    print(f"v0 MAE:       {v0_mae:.5f} m/s")
    # print(f"traj MSE full:{np.mean(mse_full_list):.6f}")
    # print(f"traj MSE obs: {np.mean(mse_obs_list):.6f}")
    # print(f"traj MSE fut: {np.nanmean(mse_future_list):.6f}")


evaluate(
    test_loader,
    model,
    titel="",
)

## 8) Demo: Rekonstruktion + Klassifikation für einen einzelnen Wurf

In [ ]:
@torch.no_grad()
def demo_reconstruction(model, ds, n_obs=None, idx=0):
    s = ds[idx]
    wurf_id = s["wurf_id"]

    t_obs = s["t_obs"].unsqueeze(0).to(device)
    xy_obs = s["xy_obs"].unsqueeze(0).to(device)

    t_min = s["t_min"].unsqueeze(0).to(device)
    t_obs_max = s["t_obs_max"].unsqueeze(0).to(device)
    denom = s["denom"].unsqueeze(0).to(device).clamp_min(1e-4)
    log_denom = torch.log(denom)

    t_n_obs = normalize_time(t_obs, t_min, denom)

    # Inferenz von (theta, v0) ausschließlich aus Beobachtungen
    out_obs = model(t_n_obs, xy_obs, t_n_obs, log_denom)
    theta_pred = out_obs["theta"]
    v0_pred = out_obs["v0"]

    # Query-Zeiten bis Korbhöhe (kein t_full als Input)
    if t_obs.shape[1] > 1:
        dt = float((t_obs[:, 1:] - t_obs[:, :-1]).mean().item())
    else:
        dt = 0.02

    y0 = xy_obs[:, 0, 1]
    t_query = make_query_times_until_rim_height(
        t_min=t_min, t_obs_max=t_obs_max, denom=denom,
        theta=theta_pred, v0=v0_pred, y0=y0,
        dt=dt, h_rim=H_RIM
    )  # [1,N]

    t_n_query = normalize_time(t_query, t_min, denom)

    out = model(t_n_obs, xy_obs, t_n_query, log_denom)
    xy_hat_t = out["xy_hat"].squeeze(0)  # [N,2]
    t_q_1d = t_query.squeeze(0)

    # Hartes Abschneiden bei Korbhöhe (verhindert unnötige Extrapolation unterhalb der Ebene)
    t_q_trim, xy_hat_trim = trim_prediction_at_rim_height(t_q_1d, xy_hat_t, h_rim=H_RIM)

    xy_hat = xy_hat_trim.detach().cpu().numpy()
    xy_obs_np = s["xy_obs"].numpy()

    # Ground Truth nur für Visualisierung (kein Input)
    xy_full = s["xy_full"].numpy()

    theta_gt = float(s["theta"].item())
    v0_gt = float(s["v0"].item())
    theta_pred_val = float(theta_pred.item())
    v0_pred_val = float(v0_pred.item())

    theta_pred_deg = math.degrees(theta_pred_val)
    theta_gt_deg = math.degrees(theta_gt)

    pred_label = int(out["logits"].argmax(dim=1).item())
    print(f"Wurf {wurf_id} | true label={s['label'].item()} | pred label={pred_label}")
    print(f"theta true={theta_gt:.4f}, pred={theta_pred_val:.4f}")
    print(f"v0    true={v0_gt:.4f}, pred={v0_pred_val:.4f}")

    plt.figure()
    plt.scatter(xy_obs_np[:, 0], xy_obs_np[:, 1], s=20, label="Beobachtungen (Eingabe)")
    plt.plot(xy_full[:, 0], xy_full[:, 1], linewidth=2, label="Referenztrajektorie")
    plt.plot(xy_hat[:, 0], xy_hat[:, 1], linewidth=2, label="Modellvorhersage")

    plt.hlines(y=3.048, xmin=4.115 - 0.2286, xmax=4.115 + 0.2286, linewidth=3, label="Korb (10ft)")

    # plt.title(
    #     f"Trajektorienvorhersage: amortisiertes PINN, N_obs={n_obs}, rauschfrei |"
    #     f"Wurf-ID={wurf_id} |"
    #     f"(GT={int(s['label'].item())}, Pred={pred_label})\n"
    #     # f"theta true={theta_gt_deg:.2f}°, pred={theta_pred_deg:.2f}° | v0 true={v0_gt:.2f}, pred={v0_pred_val:.2f}"
    # )
    print(
        f"theta true={theta_gt_deg:.2f}°, pred={theta_pred_deg:.2f}° | v0 true={v0_gt:.2f}, pred={v0_pred_val:.2f}"
    )
    plt.xlabel("x [m]"); plt.ylabel("y [m]"); plt.legend(); plt.grid(True, alpha=0.3)
    plt.xlim(-0.1, 5)
    plt.ylim(2.4, 4.15)
    plt.show()

@torch.no_grad()
def demo_reconstruction_noisy_vs_clean(model, ds_noisy, ds_clean, n_obs=None, idx=0):
    """
    ds_noisy: Dataset mit use_noisy=True (Inputs sind noisy)
    ds_clean: Dataset mit use_noisy=False (GT-Trajektorie ist clean)
    """
    # --- noisy sample ---
    sN = ds_noisy[idx]
    wurf_id = sN["wurf_id"]

    # --- find matching clean sample by wurf_id ---
    if not hasattr(demo_reconstruction_noisy_vs_clean, "_clean_index") or getattr(
        demo_reconstruction_noisy_vs_clean, "_clean_index_ds_id", None
    ) != id(ds_clean):
        clean_index = {}
        for j in range(len(ds_clean)):
            wj = ds_clean[j]["wurf_id"]
            clean_index[wj] = j
        demo_reconstruction_noisy_vs_clean._clean_index = clean_index
        demo_reconstruction_noisy_vs_clean._clean_index_ds_id = id(ds_clean)

    clean_index = demo_reconstruction_noisy_vs_clean._clean_index
    if wurf_id not in clean_index:
        raise KeyError(f"wurf_id={wurf_id} nicht im ds_clean gefunden. Prüfe Split/IDs.")

    sC = ds_clean[clean_index[wurf_id]]

    # --- model forward: nur noisy Beobachtungen, Query-Zeiten dynamisch bis Korbhöhe ---
    t_obs = sN["t_obs"].unsqueeze(0).to(device)
    xy_obs = sN["xy_obs"].unsqueeze(0).to(device)

    t_min = sN["t_min"].unsqueeze(0).to(device)
    t_obs_max = sN["t_obs_max"].unsqueeze(0).to(device)
    denom = sN["denom"].unsqueeze(0).to(device).clamp_min(1e-4)
    log_denom = torch.log(denom)

    t_n_obs = normalize_time(t_obs, t_min, denom)

    # Parameter aus Beobachtungen (kein t_full)
    out_obs = model(t_n_obs, xy_obs, t_n_obs, log_denom)
    theta_pred = out_obs["theta"]
    v0_pred = out_obs["v0"]

    if t_obs.shape[1] > 1:
        dt = float((t_obs[:, 1:] - t_obs[:, :-1]).mean().item())
    else:
        dt = 0.02

    y0 = xy_obs[:, 0, 1]
    t_query = make_query_times_until_rim_height(
        t_min=t_min, t_obs_max=t_obs_max, denom=denom,
        theta=theta_pred, v0=v0_pred, y0=y0,
        dt=dt, h_rim=H_RIM
    )  # [1,N]
    t_n_query = normalize_time(t_query, t_min, denom)

    out = model(t_n_obs, xy_obs, t_n_query, log_denom)
    xy_hat_t = out["xy_hat"].squeeze(0)  # [N,2]
    t_q_1d = t_query.squeeze(0)

    # Abschneiden bei Korbhöhe
    t_q_trim, xy_hat_trim = trim_prediction_at_rim_height(t_q_1d, xy_hat_t, h_rim=H_RIM)
    xy_hat = xy_hat_trim.detach().cpu().numpy()

    # --- arrays for plotting ---
    xy_obs_np = sN["xy_obs"].numpy()           # noisy obs
    xy_gt_clean = sC["xy_full"].numpy()        # clean GT full (nur Plot)

    theta_gt = float(sC["theta"].item())
    v0_gt = float(sC["v0"].item())
    theta_pred_val = float(theta_pred.item())
    v0_pred_val = float(v0_pred.item())

    theta_pred_deg = math.degrees(theta_pred_val)
    theta_gt_deg = math.degrees(theta_gt)

    pred_label = int(out["logits"].argmax(dim=1).item())

    plt.figure()
    plt.scatter(xy_obs_np[:, 0], xy_obs_np[:, 1], s=20, label="Beobachtungen (Eingabe)")
    plt.plot(xy_gt_clean[:, 0], xy_gt_clean[:, 1], linewidth=2, label="Referenztrajektorie")
    plt.plot(xy_hat[:, 0], xy_hat[:, 1], linewidth=2, label="Modellvorhersage")

    plt.hlines(y=3.048, xmin=4.115 - 0.2286, xmax=4.115 + 0.2286, linewidth=3, label="Korb (10ft)")

    # plt.title(
        # f"Trajektorienvorhersage: amortisiertes PINN, N_obs={n_obs}, verrauscht |"
        # f"Wurf-ID={wurf_id} |"
        # f"(GT={int(sC['label'].item())}, Pred={pred_label})\n"
        # f"theta true={theta_gt_deg:.2f}°, pred={theta_pred_deg:.2f}° | v0 true={v0_gt:.2f}, pred={v0_pred_val:.2f}"
    # )
    print(f"theta true={theta_gt_deg:.2f}°, pred={theta_pred_deg:.2f}° | v0 true={v0_gt:.2f}, pred={v0_pred_val:.2f}")
    plt.xlabel("x [m]"); plt.ylabel("y [m]"); plt.legend(); plt.grid(True, alpha=0.3)
    plt.xlim(-0.1, 5)
    plt.ylim(2.4, 4.15)
    plt.show()


@torch.no_grad()
def evaluate_trajectory_mse_vs_clean_gt(
    loader_obs,
    ds_clean,
    model,
    n_obs: int,
    compute_obs_future: bool = True,
):
    model.eval()

    # Cache: wurf_id -> clean xy_full (CPU Tensor)
    if (not hasattr(evaluate_trajectory_mse_vs_clean_gt, "_clean_cache")) or (
        getattr(evaluate_trajectory_mse_vs_clean_gt, "_clean_cache_ds_id", None) != id(ds_clean)
    ):
        cache = {}
        for i in range(len(ds_clean)):
            s = ds_clean[i]
            cache[int(s["wurf_id"])] = s["xy_full"].detach().cpu()  # [M,2]
        evaluate_trajectory_mse_vs_clean_gt._clean_cache = cache
        evaluate_trajectory_mse_vs_clean_gt._clean_cache_ds_id = id(ds_clean)

    clean_cache = evaluate_trajectory_mse_vs_clean_gt._clean_cache

    # Wir akkumulieren SSE und numel, damit der Gesamt-MSE korrekt über alle Punkte gemittelt wird.
    sse_full = 0.0
    n_full = 0

    sse_obs = 0.0
    n_obs_count = 0
    sse_future = 0.0
    n_future_count = 0

    for batch in loader_obs:
        # Inputs (noisy oder clean – je nachdem wie loader_obs gebaut ist)
        t_obs  = batch["t_obs"].to(device)      # [B,n_obs]
        xy_obs = batch["xy_obs"].to(device)     # [B,n_obs,2]
        t_full = batch["t_full"].to(device)     # [B,M]

        t_min  = batch["t_min"].to(device)      # [B]
        denom  = batch["denom"].to(device).clamp_min(1e-4)  # [B]
        log_denom = torch.log(denom)

        t_n_obs  = normalize_time(t_obs,  t_min, denom)     # [B,n_obs]
        t_n_full = normalize_time(t_full, t_min, denom)     # [B,M]

        # Modellvorhersage NUR aus Beobachtungen (kein clean xy_full wird ins Modell gegeben)
        out = model(t_n_obs, xy_obs, t_n_full, log_denom)
        pred_xy = out["xy_hat"]  # [B,M,2]

        # Clean-GT passend zu den wurf_id holen (nur Auswertung)
        ids = batch["wurf_id"]   # Liste der Länge B
        gt_list = []
        for wid in ids:
            wid_int = int(wid)
            if wid_int not in clean_cache:
                raise KeyError(f"wurf_id={wid_int} nicht im ds_clean gefunden. Prüfe Split/IDs.")
            gt_list.append(clean_cache[wid_int])
        gt_xy = torch.stack(gt_list, dim=0).to(device)  # [B,M,2]

        # Full MSE (über alle Koordinaten und Zeitpunkte)
        diff = pred_xy - gt_xy
        sse_full += float((diff * diff).sum().item())
        n_full += diff.numel()

        if compute_obs_future:
            diff_obs = diff[:, :n_obs, :]
            diff_fut = diff[:, n_obs:, :]

            sse_obs += float((diff_obs * diff_obs).sum().item())
            n_obs_count += diff_obs.numel()

            # Falls n_obs == M, ist future leer -> numel = 0
            if diff_fut.numel() > 0:
                sse_future += float((diff_fut * diff_fut).sum().item())
                n_future_count += diff_fut.numel()

    mse_full = sse_full / max(1, n_full)
    result = {"mse_full_vs_clean_gt": mse_full}

    if compute_obs_future:
        result["mse_obs_vs_clean_gt"] = sse_obs / max(1, n_obs_count)
        # Wenn future leer war, bleibt n_future_count = 0 -> NaN setzen
        result["mse_future_vs_clean_gt"] = (sse_future / n_future_count) if n_future_count > 0 else float("nan")

    print("=== Trajektorienfehler vs. clean Ground Truth ===")
    print(f"MSE full:   {result['mse_full_vs_clean_gt']:.8f}")
    if compute_obs_future:
        print(f"MSE obs:    {result['mse_obs_vs_clean_gt']:.8f}")
        print(f"MSE future: {result['mse_future_vs_clean_gt']:.8f}")

    return result

evaluate_trajectory_mse_vs_clean_gt(test_loader, ds_clean=test_ds, model=model, n_obs=30)
demo_reconstruction(model,test_ds, n_obs=30, idx=0)

## 9) Nächster Schritt: Training mit verrauschten Koordinaten

In [ ]:
train_ds_noisy = FreeThrowDataset(df, train_ids, n_obs=n_obs, use_noisy=True)
val_ds_noisy   = FreeThrowDataset(df, val_ids,   n_obs=n_obs, use_noisy=True)
test_ds_noisy  = FreeThrowDataset(df, test_ids,  n_obs=n_obs, use_noisy=True)
test_ds_clean  = FreeThrowDataset(df, test_ids,  n_obs=n_obs, use_noisy=False)

train_loader_noisy = DataLoader(train_ds_noisy, batch_size=64, shuffle=True, collate_fn=collate_fn)
val_loader_noisy   = DataLoader(val_ds_noisy,   batch_size=64, shuffle=False, collate_fn=collate_fn)
test_loader_noisy  = DataLoader(test_ds_noisy,  batch_size=64, shuffle=False, collate_fn=collate_fn)

model_noisy = AmortizedPINNClassifier(num_classes=int(df["label"].nunique())).to(device)

history_noisy = train_model(
    model_noisy,
    train_loader_noisy,
    val_loader_noisy,
    epochs=800,
    lr=2e-3,
    patience=50,
    n_col=64,
    w_data=5.0,
    w_phys=1.0,
    w_ic=10.0,
    w_cls=0.1,
)

evaluate(
    test_loader_noisy,
    model_noisy,
    titel="",
)

evaluate_trajectory_mse_vs_clean_gt(test_loader_noisy, ds_clean=test_ds_clean, model=model_noisy, n_obs=30)
demo_reconstruction_noisy_vs_clean(model_noisy,test_ds_noisy, test_ds_clean, n_obs=30, idx=11)

In [ ]:
train_ds_noisy50 = FreeThrowDataset(df, train_ids, n_obs=50, use_noisy=True)
val_ds_noisy50 = FreeThrowDataset(df, val_ids, n_obs=50, use_noisy=True)
test_ds_noisy50 = FreeThrowDataset(df, test_ids, n_obs=50, use_noisy=True)
test_ds_clean50 = FreeThrowDataset(df, test_ids, n_obs=50, use_noisy=False)

train_loader_noisy50 = DataLoader(
    train_ds_noisy50, batch_size=64, shuffle=True, collate_fn=collate_fn
)
val_loader_noisy50 = DataLoader(
    val_ds_noisy50, batch_size=64, shuffle=False, collate_fn=collate_fn
)
test_loader_noisy50 = DataLoader(
    test_ds_noisy50, batch_size=64, shuffle=False, collate_fn=collate_fn
)

model_noisy_50 = AmortizedPINNClassifier(num_classes=int(df["label"].nunique())).to(
    device
)

history_noisy = train_model(
    model_noisy_50,
    train_loader_noisy50,
    val_loader_noisy50,
    epochs=800,
    lr=2e-3,
    patience=50,
    n_col=64,
    w_data=5.0,
    w_phys=1.0,
    w_ic=10.0,
    w_cls=0.01,
)

evaluate(
    test_loader_noisy50,
    model_noisy_50,
    titel="",
)

evaluate_trajectory_mse_vs_clean_gt(
    test_loader_noisy50, ds_clean=test_ds_clean50, model=model_noisy_50, n_obs=50
)
demo_reconstruction_noisy_vs_clean(
    model_noisy_50, test_ds_noisy50, test_ds_clean50, n_obs=50, idx=14
)

In [ ]:
train_ds_noisy20 = FreeThrowDataset(df, train_ids, n_obs=20, use_noisy=True)
val_ds_noisy20 = FreeThrowDataset(df, val_ids, n_obs=20, use_noisy=True)
test_ds_noisy20 = FreeThrowDataset(df, test_ids, n_obs=20, use_noisy=True)
test_ds_clean20 = FreeThrowDataset(df, test_ids, n_obs=20, use_noisy=False)

train_loader_noisy20 = DataLoader(
    train_ds_noisy20, batch_size=64, shuffle=True, collate_fn=collate_fn
)
val_loader_noisy20 = DataLoader(
    val_ds_noisy20, batch_size=64, shuffle=False, collate_fn=collate_fn
)
test_loader_noisy20 = DataLoader(
    test_ds_noisy20, batch_size=64, shuffle=False, collate_fn=collate_fn
)

model_noisy_20 = AmortizedPINNClassifier(num_classes=int(df["label"].nunique())).to(device)

history_noisy20 = train_model(
    model_noisy_20,
    train_loader_noisy20,
    val_loader_noisy20,
    epochs=800,
    lr=2e-3,
    patience=50,
    n_col=64,
    w_data=5.0,
    w_phys=1.0,
    w_ic=10.0,
    w_cls=0.01,
)

evaluate(
    test_loader_noisy20,
    model_noisy_20,
    titel="Konfusionsmatrix: amortisiertes PINN, N_obs=20 (gekürzt), verrauscht",
)

demo_reconstruction_noisy_vs_clean(model_noisy_20,test_ds_noisy20, test_ds_clean20, n_obs=20, idx=11)

In [ ]:
train_ds_clean20 = FreeThrowDataset(df, train_ids, n_obs=20, use_noisy=False)
val_ds_clean20 = FreeThrowDataset(df, val_ids, n_obs=20, use_noisy=False)
test_ds_clean20 = FreeThrowDataset(df, test_ids, n_obs=20, use_noisy=False)

train_loader20 = DataLoader(
    train_ds_clean20, batch_size=64, shuffle=True, collate_fn=collate_fn
)
val_loader20 = DataLoader(
    val_ds_clean20, batch_size=64, shuffle=False, collate_fn=collate_fn
)
test_loader20 = DataLoader(
    test_ds_clean20, batch_size=64, shuffle=False, collate_fn=collate_fn
)

model_clean_20 = AmortizedPINNClassifier(num_classes=int(df["label"].nunique())).to(device)

history_20 = train_model(
    model_clean_20,
    train_loader20,
    val_loader20,
    epochs=800,
    lr=2e-3,
    patience=50,
    n_col=64,
    w_data=5.0,
    w_phys=1.0,
    w_ic=10.0,
    w_cls=0.01,
)

evaluate(
    test_loader20,
    model_clean_20,
    titel="Konfusionsmatrix: amortisiertes PINN, N_obs=20 (gekürzt), rauschfrei",
)

demo_reconstruction(
    model_clean_20, test_ds_clean20, n_obs=20, idx=11
)

In [ ]:
train_ds_noisy10 = FreeThrowDataset(df, train_ids, n_obs=10, use_noisy=True)
val_ds_noisy10 = FreeThrowDataset(df, val_ids, n_obs=10, use_noisy=True)
test_ds_noisy10 = FreeThrowDataset(df, test_ids, n_obs=10, use_noisy=True)
test_ds_clean10 = FreeThrowDataset(df, test_ids, n_obs=10, use_noisy=False)

train_loader_noisy10 = DataLoader(
    train_ds_noisy10, batch_size=64, shuffle=True, collate_fn=collate_fn
)
val_loader_noisy10 = DataLoader(
    val_ds_noisy10, batch_size=64, shuffle=False, collate_fn=collate_fn
)
test_loader_noisy10 = DataLoader(
    test_ds_noisy10, batch_size=64, shuffle=False, collate_fn=collate_fn
)

model_noisy_10 = AmortizedPINNClassifier(num_classes=int(df["label"].nunique())).to(device)

history_noisy10 = train_model(
    model_noisy_10,
    train_loader_noisy10,
    val_loader_noisy10,
    epochs=800,
    lr=2e-3,
    patience=50,
    n_col=64,
    w_data=5.0,
    w_phys=1.0,
    w_ic=10.0,
    w_cls=0.01,
)

evaluate(
    test_loader_noisy10,
    model_noisy_10,
    titel="Konfusionsmatrix: amortisiertes PINN, N_obs=10 (gekürzt), verrauscht",
)

demo_reconstruction_noisy_vs_clean(
    model_noisy_10, test_ds_noisy10, test_ds_clean10, n_obs=10, idx=11
)

In [ ]:
train_ds_clean_10 = FreeThrowDataset(df, train_ids, n_obs=10, use_noisy=False)
val_ds_clean_10 = FreeThrowDataset(df, val_ids, n_obs=10, use_noisy=False)
test_ds_clean_10 = FreeThrowDataset(df, test_ids, n_obs=10, use_noisy=False)

train_loader10 = DataLoader(train_ds_clean_10, batch_size=64, shuffle=True, collate_fn=collate_fn)
val_loader10 = DataLoader(val_ds_clean_10, batch_size=64, shuffle=False, collate_fn=collate_fn)
test_loader10 = DataLoader(test_ds_clean_10, batch_size=64, shuffle=False, collate_fn=collate_fn)

model_clean_10 = AmortizedPINNClassifier(num_classes=int(df["label"].nunique())).to(device)

history_10 = train_model(
    model_clean_10,
    train_loader10,
    val_loader10,
    epochs=800,
    lr=2e-3,
    patience=50,
    n_col=64,
    w_data=5.0,
    w_phys=1.0,
    w_ic=10.0,
    w_cls=0.01,
)

evaluate(
    test_loader10,
    model_clean_10,
    titel="Konfusionsmatrix: amortisiertes PINN, N_obs=10 (gekürzt), rauschfrei",
)

demo_reconstruction(model_clean_10, test_ds_clean_10, n_obs=10, idx=11)

In [ ]:
@torch.no_grad()
def evaluate_param_estimation(
    loader,
    model,
    device=device,
    theta_in_degrees: bool = True,
    return_dataframe: bool = True,
    verbose: bool = True,
):
    model.eval()

    records = []
    for batch in loader:
        # Beobachtungen
        t_obs = batch["t_obs"].to(device)  # [B, n_obs]
        xy_obs = batch["xy_obs"].to(device)  # [B, n_obs, 2]

        # Zeit-Normalisierungskontext (ausschließlich aus Beobachtungsfenster)
        t_min = batch["t_min"].to(device)  # [B]
        denom = batch["denom"].to(device).clamp_min(1e-4)  # [B]
        log_denom = torch.log(denom)

        # normalisierte Zeitpunkte der Beobachtungen (Query = Obs, da hier nur Parameter interessiert)
        t_n_obs = normalize_time(t_obs, t_min, denom)

        # Inferenz
        out = model(t_n_obs, xy_obs, t_n_obs, log_denom)
        theta_hat = out["theta"].detach().cpu().numpy()
        v0_hat = out["v0"].detach().cpu().numpy()

        theta_gt = batch["theta"].detach().cpu().numpy()
        v0_gt = batch["v0"].detach().cpu().numpy()

        # optional: IDs (collate_fn liefert hier eine Python-Liste)
        wurf_ids = batch.get("wurf_id", None)
        if wurf_ids is None:
            wurf_ids = [None] * len(theta_gt)

        for i in range(len(theta_gt)):
            th_err = float(theta_hat[i] - theta_gt[i])
            v0_err = float(v0_hat[i] - v0_gt[i])
            records.append(
                {
                    "wurf_id": int(wurf_ids[i]) if wurf_ids[i] is not None else None,
                    "theta_gt_rad": float(theta_gt[i]),
                    "theta_hat_rad": float(theta_hat[i]),
                    "theta_err_rad": th_err,
                    "theta_abs_err_rad": abs(th_err),
                    "v0_gt": float(v0_gt[i]),
                    "v0_hat": float(v0_hat[i]),
                    "v0_err": v0_err,
                    "v0_abs_err": abs(v0_err),
                }
            )

    df_err = pd.DataFrame.from_records(records)

    def _stats(err: np.ndarray) -> dict:
        err = np.asarray(err, dtype=np.float64)
        abs_err = np.abs(err)
        out = {
            "mae": float(abs_err.mean()),
            "rmse": float(np.sqrt(np.mean(err**2))),
            "bias": float(err.mean()),
            "std": float(err.std()),
            "median_ae": float(np.median(abs_err)),
            "q90_ae": float(np.quantile(abs_err, 0.90)),
        }
        return out

    theta_err = df_err["theta_err_rad"].to_numpy()
    v0_err = df_err["v0_err"].to_numpy()

    metrics = {
        "theta_rad": _stats(theta_err),
        "v0": _stats(v0_err),
    }

    # relative Fehler für v0 (stabil gegen sehr kleine v0)
    v0_gt_safe = np.clip(df_err["v0_gt"].to_numpy(dtype=np.float64), 1e-9, None)
    metrics["v0"]["rel_mae"] = float(
        (df_err["v0_abs_err"].to_numpy(dtype=np.float64) / v0_gt_safe).mean()
    )
    metrics["v0"]["rel_rmse"] = float(np.sqrt(np.mean((v0_err / v0_gt_safe) ** 2)))

    # Korrelationen (Validierung, dass Ranking/Trend stimmt; NaN bei zu wenigen Samples)
    if len(df_err) >= 2:
        metrics["theta_rad"]["pearson_r"] = float(
            np.corrcoef(df_err["theta_gt_rad"], df_err["theta_hat_rad"])[0, 1]
        )
        metrics["v0"]["pearson_r"] = float(
            np.corrcoef(df_err["v0_gt"], df_err["v0_hat"])[0, 1]
        )
    else:
        metrics["theta_rad"]["pearson_r"] = float("nan")
        metrics["v0"]["pearson_r"] = float("nan")

    # Grad-Ansicht für theta (optional, oft interpretierbarer)
    if theta_in_degrees:
        deg = 180.0 / np.pi
        df_err["theta_gt_deg"] = df_err["theta_gt_rad"] * deg
        df_err["theta_hat_deg"] = df_err["theta_hat_rad"] * deg
        df_err["theta_err_deg"] = df_err["theta_err_rad"] * deg
        df_err["theta_abs_err_deg"] = df_err["theta_abs_err_rad"] * deg

        metrics["theta_deg"] = {
            k: (v * deg if k != "pearson_r" else v)
            for k, v in metrics["theta_rad"].items()
        }

    if verbose:
        if theta_in_degrees:
            th = metrics["theta_deg"]
            print(
                f"theta: MAE={th['mae']:.3f}° | RMSE={th['rmse']:.3f}° | Bias={th['bias']:.3f}° | "
                f"MedianAE={th['median_ae']:.3f}° | Q90AE={th['q90_ae']:.3f}° | r={th['pearson_r']:.3f}"
            )
        else:
            th = metrics["theta_rad"]
            print(
                f"theta: MAE={th['mae']:.5f} rad | RMSE={th['rmse']:.5f} rad | Bias={th['bias']:.5f} rad | "
                f"MedianAE={th['median_ae']:.5f} rad | Q90AE={th['q90_ae']:.5f} rad | r={th['pearson_r']:.3f}"
            )

        vv = metrics["v0"]
        print(
            f"v0:    MAE={vv['mae']:.4f} m/s | RMSE={vv['rmse']:.4f} m/s | Bias={vv['bias']:.4f} m/s | "
            f"MedianAE={vv['median_ae']:.4f} m/s | Q90AE={vv['q90_ae']:.4f} m/s | "
            f"relMAE={100*vv['rel_mae']:.2f}% | r={vv['pearson_r']:.3f}"
        )

    if return_dataframe:
        return metrics, df_err
    return metrics

In [ ]:
print("=== Parameterfehler (theta, v0) auf Testset (rauschfrei, N_obs=30) ===")

metrics, df_err = evaluate_param_estimation(
    loader=test_loader,
    model=model,
    device=device,
    theta_in_degrees=True,
    return_dataframe=True,
    verbose=True,
)

print("=== Parameterfehler (theta, v0) auf Testset (verrauscht, N_obs=30) ===")
metrics_noisy, df_err_noisy = evaluate_param_estimation(
    loader=test_loader_noisy,
    model=model_noisy,
    device=device,
    theta_in_degrees=True,
    return_dataframe=True,
    verbose=True,
)

print("=== Parameterfehler (theta, v0) auf Testset (rauschfrei, N_obs=20) ===")

metrics20, df_err20 = evaluate_param_estimation(
    loader=test_loader20,
    model=model_clean_20,
    device=device,
    theta_in_degrees=True,
    return_dataframe=True,
    verbose=True,
)

print("=== Parameterfehler (theta, v0) auf Testset (verrauscht, N_obs=20) ===")
metrics_noisy20, df_err_noisy20 = evaluate_param_estimation(
    loader=test_loader_noisy20,
    model=model_noisy_20,
    device=device,
    theta_in_degrees=True,
    return_dataframe=True,
    verbose=True,
)

print("=== Parameterfehler (theta, v0) auf Testset (rauschfrei, N_obs=10) ===")

metrics10, df_err10 = evaluate_param_estimation(
    loader=test_loader10,
    model=model_clean_10,
    device=device,
    theta_in_degrees=True,
    return_dataframe=True,
    verbose=True,
)

print("=== Parameterfehler (theta, v0) auf Testset (verrauscht, N_obs=10) ===")
metrics_noisy10, df_err_noisy10 = evaluate_param_estimation(
    loader=test_loader_noisy10,
    model=model_noisy_10,
    device=device,
    theta_in_degrees=True,
    return_dataframe=True,
    verbose=True,
)